# PDF Processing & Vertex AI Search Pipeline
This notebook implements the end-to-end pipeline:
1. Extract images from PDFs using Document AI Layout Parser.
2. Describe images using Gemini 1.5 Pro.
3. Mask the original images in the PDFs using PyMuPDF.
4. Upload cleaned PDFs and Custom Chunk JSONL to GCS.
5. Create a Vertex AI Search Data Store and Search App.

In [ ]:
!pip install google-cloud-documentai google-cloud-storage google-cloud-discoveryengine PyMuPDF google-generativeai vertexai requests google-auth -q

### Step 0: Configuration & Authentication
Please ensure you fill in the placeholder values below before running the rest of the cells.

In [ ]:
import time
import google.auth
from google.auth.transport.requests import Request

# --- CONFIGURATION ---
PROJECT_ID = "gemini-ent-agent-demos"  # @param type/string
LOCATION = "global"

# Using a timestamp to genericize the creation of new resources
timestamp = int(time.time())

DOCAI_PROCESSOR_ID = f"pdf-pipeline-layout-parser-{timestamp}" 
DOCAI_LOCATION = "us" # Usually 'us' or 'eu'

GCS_BUCKET_NAME = f"{PROJECT_ID}-pdf-pipeline-{timestamp}"
DATA_STORE_ID = f"pdf-pipeline-ds-{timestamp}"
ENGINE_ID = f"pdf-pipeline-app-{timestamp}"

# Automatically get the GCP access token
creds, project = google.auth.default()
auth_req = Request()
creds.refresh(auth_req)

print(f"Using Project ID: {PROJECT_ID}")
print(f"GCS Bucket: {GCS_BUCKET_NAME}")
print(f"Data Store ID: {DATA_STORE_ID}")
print(f"Engine ID: {ENGINE_ID}")


### Phase 1: Document Processing (Doc AI, Gemini, PyMuPDF)
We will locate the PDFs in `staging/`, use Doc AI to find images, describe them with Gemini, and mask them.

In [ ]:
import os
import fitz  # PyMuPDF
from google.cloud import documentai
import vertexai
from vertexai.generative_models import GenerativeModel, Part, Image as VertexImage

# Initialize Vertex AI for Gemini
vertexai.init(project=PROJECT_ID, location="us-central1") # Gemini typically runs in us-central1
model = GenerativeModel("gemini-1.5-pro-001")

STAGING_DIR = "staging"
PROCESSED_DIR = "processed"
IMAGES_DIR = "extracted_images"
os.makedirs(PROCESSED_DIR, exist_ok=True)
os.makedirs(IMAGES_DIR, exist_ok=True)

# Document AI Setup
docai_client = documentai.DocumentProcessorServiceClient()
processor_name = docai_client.processor_path(PROJECT_ID, DOCAI_LOCATION, DOCAI_PROCESSOR_ID)

pdf_files = [f for f in os.listdir(STAGING_DIR) if f.endswith('.pdf')]
print(f"Found PDFs: {pdf_files}")

metadata_records = []

for pdf_filename in pdf_files:
    pdf_path = os.path.join(STAGING_DIR, pdf_filename)
    print(f"\nProcessing: {pdf_filename}")
    
    # 1. Document AI Layout Parsing
    with open(pdf_path, "rb") as f:
        image_content = f.read()
    raw_document = documentai.RawDocument(content=image_content, mime_type="application/pdf")
    request = documentai.ProcessRequest(name=processor_name, raw_document=raw_document)
    print("Calling Document AI...")
    try:
        result = docai_client.process_document(request=request)
    except Exception as e:
        print(f"Error calling Doc AI. Did you set DOCAI_PROCESSOR_ID? {e}")
        continue
        
    document = result.document
    
    # 2. Extract Images & Call Gemini
    doc_fitz = fitz.open(pdf_path)
    
    for page in document.pages:
        page_num = page.page_number
        fitz_page = doc_fitz[page_num - 1]
        
        for image_block in page.image_blocks:
            # Get bounding box
            vertices = image_block.layout.bounding_poly.normalized_vertices
            if not vertices:
                continue
                
            x_coords = [v.x for v in vertices]
            y_coords = [v.y for v in vertices]
            min_x, max_x = min(x_coords), max(x_coords)
            min_y, max_y = min(y_coords), max(y_coords)
            
            rect = fitz.Rect(
                min_x * fitz_page.rect.width, 
                min_y * fitz_page.rect.height,
                max_x * fitz_page.rect.width, 
                max_y * fitz_page.rect.height
            )
            
            # Save temporary image
            pix = fitz_page.get_pixmap(clip=rect)
            img_filename = f"{pdf_filename}_p{page_num}_{int(time.time())}.png"
            img_path = os.path.join(IMAGES_DIR, img_filename)
            pix.save(img_path)
            
            # Call Gemini
            vertex_img = VertexImage.load_from_file(img_path)
            try:
                print(f"Calling Gemini for image on page {page_num}...")
                response = model.generate_content([vertex_img, "Describe the content of this image in detail."])
                description = response.text
            except Exception as e:
                print(f"Gemini error: {e}")
                description = "Image description not available."
                
            # Store metadata
            chunk_id = f"chunk_{img_filename}"
            metadata_records.append({
                "chunk_id": chunk_id,
                "pdf_filename": pdf_filename,
                "page_number": page_num,
                "description": description
            })
            
            # 3. Mask Image in PDF
            fitz_page.draw_rect(rect, color=(0,0,0), fill=(0,0,0)) # Black box over image
            
    # Save Cleaned PDF
    cleaned_pdf_path = os.path.join(PROCESSED_DIR, pdf_filename)
    doc_fitz.save(cleaned_pdf_path)
    doc_fitz.close()
    print(f"Saved cleaned PDF to {cleaned_pdf_path}")


### Phase 2: Metadata Creation & GCS Upload
We will write the custom chunk JSONL file and upload everything to a new GCS bucket.

In [ ]:
import json
from google.cloud import storage

# Generate JSONL for Vertex AI Search
jsonl_filename = "custom_chunks.jsonl"
with open(jsonl_filename, 'w') as f:
    for record in metadata_records:
        # Vertex AI Search Custom Chunk format
        chunk_data = {
            "id": record["chunk_id"],
            "jsonData": json.dumps({
                "chunkedDocument": {
                    "chunks": [
                        {
                            "chunkId": record["chunk_id"],
                            "content": record["description"],
                            # If you wanted to attach the blob (image) itself, you would do it here
                            # similar to the byoc_guide.ipynb. For now we just use the description.
                            "pageSpan": {
                                "pageStart": record["page_number"],
                                "pageEnd": record["page_number"]
                            }
                        }
                    ]
                }
            }),
            "content": {"mimeType": "text/plain", "uri": f"gs://{GCS_BUCKET_NAME}/{record['pdf_filename']}"} 
        }
        f.write(json.dumps(chunk_data) + '\n')

print(f"Created {jsonl_filename}")

# Upload to GCS
storage_client = storage.Client(project=PROJECT_ID)
try:
    bucket = storage_client.create_bucket(GCS_BUCKET_NAME)
    print(f"Created bucket {GCS_BUCKET_NAME}")
except Exception as e:
    print(f"Bucket creation error or exists: {e}")
    bucket = storage_client.get_bucket(GCS_BUCKET_NAME)

# Upload PDFs
for pdf_filename in os.listdir(PROCESSED_DIR):
    blob = bucket.blob(pdf_filename)
    blob.upload_from_filename(os.path.join(PROCESSED_DIR, pdf_filename))
    print(f"Uploaded {pdf_filename} to GCS")

# Upload JSONL
blob = bucket.blob(jsonl_filename)
blob.upload_from_filename(jsonl_filename)
print(f"Uploaded {jsonl_filename} to GCS")


### Phase 3: Vertex AI Search Setup
Create the Data Store and Engine, then import the data.

In [ ]:
from google.cloud import discoveryengine_v1alpha as discoveryengine
from google.api_core.client_options import ClientOptions

client_options = (ClientOptions(api_endpoint=f"{LOCATION}-discoveryengine.googleapis.com") if LOCATION != "global" else None)
ds_client = discoveryengine.DataStoreServiceClient(client_options=client_options)
parent = f"projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection"

# 1. Create Data Store
data_store = discoveryengine.DataStore(
    display_name="PDF Pipeline Store",
    industry_vertical=discoveryengine.IndustryVertical.GENERIC,
    solution_types=[discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH],
    content_config=discoveryengine.DataStore.ContentConfig.CONTENT_REQUIRED,
)
request = discoveryengine.CreateDataStoreRequest(parent=parent, data_store_id=DATA_STORE_ID, data_store=data_store)
try:
    print("Creating Data Store...")
    response = ds_client.create_data_store(request=request).result()
    print(f"Created Data Store: {response.name}")
except Exception as e:
    print(f"Error: {e}")

# 2. Create Engine
engine_client = discoveryengine.EngineServiceClient(client_options=client_options)
engine = discoveryengine.Engine(
    display_name="PDF Pipeline App",
    solution_type=discoveryengine.SolutionType.SOLUTION_TYPE_SEARCH,
    data_store_ids=[DATA_STORE_ID],
    search_engine_config=discoveryengine.Engine.SearchEngineConfig(
        search_tier=discoveryengine.SearchTier.SEARCH_TIER_ENTERPRISE,
        search_add_ons=[discoveryengine.SearchAddOn.SEARCH_ADD_ON_LLM],
    ),
)
request = discoveryengine.CreateEngineRequest(parent=parent, engine_id=ENGINE_ID, engine=engine)
try:
    print("Creating Engine...")
    response = engine_client.create_engine(request=request).result()
    print(f"Created Engine: {response.name}")
except Exception as e:
    print(f"Error: {e}")

print("\nNOTE: Data Import is required next. You can import the GCS bucket via the Cloud Console, or add the Python SDK import logic here.")
print(f"Remember to import from: gs://{GCS_BUCKET_NAME}/custom_chunks.jsonl (using the unstructured/custom chunk import format)")


### Phase 4: Test Query via Answer API
We can test the Search App using the Vertex AI Search Answer API to ensure it retrieves our custom chunks.

In [ ]:
import requests

def query_answer_rest():
    creds, _ = google.auth.default()
    auth_req = google.auth.transport.requests.Request()
    creds.refresh(auth_req)
    access_token = creds.token

    base_url = f"https://{LOCATION}-discoveryengine.googleapis.com" if LOCATION != "global" else "https://discoveryengine.googleapis.com"
    url = f"{base_url}/v1alpha/projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/engines/{ENGINE_ID}/servingConfigs/default_serving_config:answer"

    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json"
    }

    # You can change this query based on the contents of your PDFs
    query_text = "What is shown in the image?"

    payload = {
        "query": { "text": query_text },
        "answerGenerationSpec": {
            "includeCitations": True,
            "modelSpec": { "modelVersion": "stable" }
        }
    }

    print(f"\nSending answer query: '{query_text}'...")
    response = requests.post(url, headers=headers, json=payload)

    if response.status_code == 200:
        data = response.json()
        answer = data.get("answer", {})
        print(f"\nAnswer Output:\n{answer.get('answerText', 'No text generated')}\n")
        
        citations = answer.get("citations", [])
        for citation in citations:
            for source in citation.get("sources", []):
                chunk_info = source.get("chunkInfo", {})
                print("--- Citation Source Found ---")
                print(f"Custom Chunk Content: '{chunk_info.get('content', '')}'")
    else:
        print(f"\n[ERROR {response.status_code}]: {response.text}")

print("NOTE: Run this only AFTER the data import has completed indexing.")
# Uncomment to test query once indexing is done
# query_answer_rest()


### Phase 5: Frontend Validation
Once the data is indexed, update your `frontend/.env.local` with these values:
```
NEXT_PUBLIC_DATA_STORE_ID="{DATA_STORE_ID}"
NEXT_PUBLIC_ENGINE_ID="{ENGINE_ID}"
```
Then run `npm run dev` in the `frontend` folder to test the UI.

### Phase 6: Resource Cleanup
Run the cell below to delete the resources created during this notebook run for easy testing. This ensures ephemeral resources don't stick around in your project and cost money.

> **Note:** The cleanup execution call is commented out at the bottom to prevent accidental deletion if you click "Run All".

In [ ]:
import requests

def clean_up_resources():
    print("Starting resource cleanup...")
    
    # 1. Delete GCS Bucket and its contents
    storage_client = storage.Client(project=PROJECT_ID)
    try:
        bucket = storage_client.get_bucket(GCS_BUCKET_NAME)
        for blob in bucket.list_blobs():
            blob.delete()
            print(f"Deleted blob: {blob.name}")
        bucket.delete()
        print(f"Deleted bucket: {GCS_BUCKET_NAME}")
    except Exception as e:
        print(f"Error deleting bucket (it may not exist): {e}")

    # 2. Authenticated REST deletion for Vertex AI Search
    creds, _ = google.auth.default()
    auth_req = google.auth.transport.requests.Request()
    creds.refresh(auth_req)
    access_token = creds.token
    headers = {"Authorization": f"Bearer {access_token}"}

    base_url = f"https://{LOCATION}-discoveryengine.googleapis.com" if LOCATION != "global" else "https://discoveryengine.googleapis.com"
    
    # Trigger Engine Delete
    print(f"\nDeleting Engine: {ENGINE_ID}")
    engine_url = f"{base_url}/v1alpha/projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/engines/{ENGINE_ID}"
    try:
        res = requests.delete(engine_url, headers=headers)
        if res.status_code == 200:
            print("Engine delete initiated successfully (LRO).")
        else:
            print(f"Engine delete response code: {res.status_code}. Message: {res.text}")
    except Exception as e:
        print(f"Error deleting engine: {e}")

    # Trigger Data Store Delete
    print(f"\nDeleting Data Store: {DATA_STORE_ID}")
    ds_url = f"{base_url}/v1alpha/projects/{PROJECT_ID}/locations/{LOCATION}/collections/default_collection/dataStores/{DATA_STORE_ID}"
    try:
        res = requests.delete(ds_url, headers=headers)
        if res.status_code == 200:
            print("Data Store delete initiated successfully (LRO).")
        else:
            print(f"Data Store delete response code: {res.status_code}. Message: {res.text}")
    except Exception as e:
        print(f"Error deleting data store: {e}")

    print("\nCleanup operations issued.")

# --- DANGER ZONE ---
# Uncomment the line below to execute the cleanup
# clean_up_resources()
